# UCI Adult — EDA exploratória

Nesta etapa exploramos todas as variáveis, sua qualidade, relação com a renda e diferenças entre os grupos de auditoria. Ainda não treinamos modelos nem escolhemos limiares.

## Perguntas-guia da EDA

1. Quais variáveis parecem ter maior associação com o alvo?
2. As diferenças por sexo e raça permanecem quando consideramos educação, ocupação e horas trabalhadas?
3. Existem categorias raras ou proxies que exigem tratamento antes do baseline?
4. O split oficial apresenta diferenças relevantes de composição?

As perguntas serão retomadas ao longo do notebook. Cada resposta será registrada como `Insight:` após a evidência correspondente.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'pt-BR' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from adult_data import load_adult
from eda_report import write_eda_outputs
frame = load_adult(ROOT / 'data' / 'raw' / 'adult_uci')
outputs = write_eda_outputs(frame, ROOT / 'data' / 'processed' / 'eda_initial')
frame.shape

(48842, 16)

## 1. Visão geral das features

A tabela mostra tipo, cardinalidade, ausência e papel analítico de cada variável.

**Pergunta 3:** existem categorias raras ou proxies que exigem tratamento antes do baseline?

In [2]:
feature_overview = pd.read_csv(outputs['feature_overview'])
feature_overview

,feature,dtype,n_unique,missing_count,missing_rate,role
0,age,int64,74,0,0.000000,predictor_candidate
1,workclass,string,8,2799,0.057307,predictor_candidate
2,fnlwgt,int64,28523,0,0.000000,predictor_candidate
3,education,string,16,0,0.000000,predictor_candidate
4,education_num,int64,16,0,0.000000,predictor_candidate
5,marital_status,string,7,0,0.000000,predictor_candidate
6,occupation,string,14,2809,0.057512,predictor_candidate
7,relationship,string,6,0,0.000000,predictor_candidate
8,race,string,5,0,0.000000,audit
9,sex,string,2,0,0.000000,audit


**Insight:** `workclass` e `occupation` apresentam aproximadamente 5,7% de valores ausentes, enquanto `native_country` apresenta aproximadamente 1,75%. Essas ausências precisarão de tratamento definido antes do baseline.

**Insight:** categorias muito pequenas, como `Never-worked` e algumas categorias de `workclass`, devem ser tratadas com cautela. Além disso, `relationship`, `marital_status`, `occupation` e `native_country` podem carregar informação proxy de atributos de auditoria; a associação será investigada antes do baseline.

## 2. Variáveis categóricas e o alvo

A taxa positiva é a proporção de registros com renda anual acima de US$ 50 mil dentro de cada categoria. Ela descreve associação, não causalidade.

**Pergunta 1:** quais variáveis categóricas parecem ter maior associação com o alvo?

In [3]:
categorical_target_rates = pd.read_csv(outputs['categorical_target_rates'])
categorical_target_rates.sort_values(['feature', 'positive_rate'], ascending=[True, False])

,feature,category,count,positive_rate
23,education,Prof-school,834,0.739808
19,education,Doctorate,594,0.725589
21,education,Masters,2657,0.549116
18,education,Bachelors,8025,0.412835
16,education,Assoc-acdm,1601,0.257964
...,...,...,...,...
6,workclass,State-gov,1981,0.267542
3,workclass,Private,33906,0.217867
7,workclass,Without-pay,21,0.095238
8,workclass,NaN,2799,0.094677


**Insight:** as taxas positivas variam entre categorias de educação, ocupação, estado civil e relacionamento. Essas diferenças indicam associação potencial com `income`, mas podem refletir composição social e econômica, não efeito causal das categorias.

## 3. Variáveis numéricas por classe

Médias, medianas e quartis ajudam a identificar diferenças de distribuição entre as classes, sem assumir que essas diferenças sejam suficientes para uma boa previsão.

**Pergunta 1:** quais variáveis numéricas parecem ter maior associação com o alvo?

In [4]:
numeric_by_target = pd.read_csv(outputs['numeric_by_target'])
numeric_by_target

,feature,income,count,mean,median,q25,q75
0,age,<=50K,37155,36.872184,34.0,25.0,46.0
1,age,>50K,11687,44.275178,43.0,36.0,51.0
2,fnlwgt,<=50K,37155,190039.565523,178811.0,117210.0,238917.0
3,fnlwgt,>50K,11687,188470.574570,176729.0,118942.5,233505.0
4,education_num,<=50K,37155,9.598493,9.0,9.0,10.0
5,education_num,>50K,11687,11.602807,12.0,10.0,13.0
6,capital_gain,<=50K,37155,147.010308,0.0,0.0,0.0
7,capital_gain,>50K,11687,4042.239497,0.0,0.0,0.0
8,capital_loss,<=50K,37155,54.151931,0.0,0.0,0.0
9,capital_loss,>50K,11687,193.528964,0.0,0.0,0.0


**Insight:** registros com renda `>50K` têm, em média, maior idade, maior escolaridade numérica e mais horas trabalhadas por semana. `fnlwgt` apresenta médias próximas entre as classes. Isso orienta a investigação, mas não define sozinho quais variáveis serão selecionadas.

## 4. Auditoria inicial por sexo, raça e interseção

Sexo e raça permanecem atributos de auditoria. A análise não os transforma automaticamente em atributos causais nem decide ainda se entrarão nas features.

**Pergunta 2:** as diferenças por sexo e raça permanecem quando consideramos educação, ocupação e horas trabalhadas?

In [5]:
categorical_target_rates.query("feature in ['sex', 'race']")

frame.assign(income_rate=(frame['income'] == '>50K').astype(int)).groupby(['sex', 'race'], dropna=False)['income_rate'].agg(['size', 'mean']).reset_index()

,sex,race,size,mean
0,Female,Amer-Indian-Eskimo,185,0.081081
1,Female,Asian-Pac-Islander,517,0.133462
2,Female,Black,2308,0.057192
3,Female,Other,155,0.070968
4,Female,White,13027,0.118370
5,Male,Amer-Indian-Eskimo,285,0.140351
6,Male,Asian-Pac-Islander,1002,0.339321
7,Male,Black,2377,0.182583
8,Male,Other,251,0.155378
9,Male,White,28735,0.315469


**Insight:** a taxa de renda `>50K` é 30,4% entre homens e 10,9% entre mulheres. Por raça, é 25,4% entre pessoas classificadas como `White` e 12,1% entre pessoas classificadas como `Black`. A tabela interseccional mostra que essas diferenças também variam dentro das combinações de sexo e raça. A persistência após considerar educação, ocupação e horas trabalhadas será testada em análises condicionais posteriores.

## 4.1 Análise condicional por estratos

A pergunta agora é se as diferenças observadas por sexo e raça continuam dentro de faixas comparáveis de escolaridade, ocupação e horas trabalhadas. A tabela usa taxas descritivas dentro de cada estrato; ela reduz diferenças de composição, mas não identifica efeito causal e pode ficar instável em grupos pequenos.

In [ ]:
conditional_target_rates = pd.read_csv(outputs['conditional_target_rates'])
conditional_target_rates.sort_values(['control', 'audit_column', 'stratum', 'positive_rate'])

### Insights:

As diferenças não desaparecem dentro dos estratos observáveis. A maior diferença entre grupos dentro de uma mesma faixa foi de 30,8 pontos percentuais por sexo nas faixas de escolaridade, 33,2 pontos por sexo entre ocupações e 22,0 pontos por sexo entre faixas de horas. Para raça, os maiores gaps foram 16,0, 38,5 e 39,9 pontos percentuais, respectivamente. Esses resultados são associações condicionais descritivas, não prova de causalidade; as células pequenas exigem intervalos de incerteza na auditoria de modelos.

## 4.2 Categorias raras e possíveis proxies

Para orientar o pré-processamento, marcamos como rara uma categoria com menos de 100 registros e como possível proxy uma diferença de composição de pelo menos 20 pontos percentuais em relação à distribuição geral de sexo ou raça. Esses limiares são triagem exploratória, não uma conclusão sobre causalidade.

In [ ]:
category_risk_diagnostics = pd.read_csv(outputs['category_risk_diagnostics'])
category_risk_diagnostics.query('is_rare or sex_proxy_flag or race_proxy_flag').sort_values(['feature', 'category'])

### Insights:

A triagem encontrou 31 categorias raras, 22 categorias com forte desequilíbrio de composição por sexo e 18 por raça. Ocupação, relacionamento, estado civil e país de origem aparecem como candidatos a carregar informação proxy. Isso não demonstra que sejam proxies causais, mas justifica manter categorias raras explícitas, documentar a imputação e auditar o comportamento por grupo antes de qualquer decisão de modelo.

## 5. Comparação entre os splits oficiais

O split oficial é preservado. Esta comparação é descritiva e não usa o teste para ajustar decisões.

**Pergunta 4:** o split oficial apresenta diferenças relevantes de composição?

In [6]:
pd.read_csv(outputs['split_comparison'])

,split,rows,positive_rate,missing_cells,missing_rate_cells,duplicate_rows
0,test_official,16281,0.236226,2203,0.009665,5
1,train_official,32561,0.240810,4262,0.009349,24


**Insight:** os splits oficiais têm taxas positivas próximas: 24,1% no treino e 23,6% no teste. A taxa de ausência também é semelhante. Não há evidência inicial de uma mudança forte na prevalência do alvo, mas a composição completa das categorias continuará sendo monitorada sem usar o teste para decisões.

## 5.1 Diferença de composição por feature

Além da prevalência e da ausência, comparamos a distribuição das categorias pela distância de variação total e as variáveis numéricas por diferença padronizada de médias. A comparação é descritiva e não transforma o teste em conjunto de desenvolvimento.

In [ ]:
split_distribution_drift = pd.read_csv(outputs['split_distribution_drift'])
split_distribution_drift.sort_values('value', key=lambda values: values.abs(), ascending=False)

### Insights:

As diferenças de composição são pequenas neste primeiro diagnóstico: a maior distância de variação total categórica foi 0,0118 para `occupation`, e o maior módulo da diferença padronizada numérica foi aproximadamente 0,0136 para `age`. Isso apoia o uso do treino oficial para desenvolvimento e do teste como avaliação final, sem eliminar a necessidade de verificar estabilidade por grupo.

## Encaminhamento

As perguntas 1 e 4 já têm respostas descritivas iniciais. As perguntas 2 e 3 exigem aprofundamento no baseline: análise condicional, investigação de proxies e avaliação de estabilidade antes da seleção do modelo.